In [3]:
import numpy as np
import pandas as pd
import re
df = pd.read_csv('/content/phishing_email.csv')
print(df.shape)

(82486, 2)


In [4]:
df.head(10)

,text_combined,label
0,hpl nom may 25 2001 see attached file hplno 52...,0
1,nom actual vols 24 th forwarded sabrae zajac h...,0
2,enron actuals march 30 april 1 201 estimated a...,0
3,hpl nom may 30 2001 see attached file hplno 53...,0
4,hpl nom june 1 2001 see attached file hplno 60...,0
5,hpl nom may 31 2001 see attached file hplno 53...,0
6,9760 tried get fancy address came back forward...,0
7,hpl noms february 15 2000 see attached file hp...,0
8,fw pooling contract template original message ...,0
9,hpl nom march 28 2000 see attached file hplo 3...,0


In [6]:
df.isnull().sum()


,0
text_combined,0
label,0


In [7]:
df['label'] = df['label'].astype(int)

In [8]:
def clean_email_text(text: str) -> str:
    text = text.lower()
    text = re.sub(r'http\S+|www\S+|https\S+', ' url ', text)
    text = re.sub(r'\S+@\S+', ' email ', text)
    text = re.sub(r'\d+', ' number ', text)
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [9]:
df['clean_text'] = df['text_combined'].apply(clean_email_text)
print('successful')

successful


In [16]:
df.drop('text_combined', axis=1, inplace=True)
df

,label,clean_text
0,0,hpl nom may number number see attached file hp...
1,0,nom actual vols number th forwarded sabrae zaj...
2,0,enron actuals march number april number number...
3,0,hpl nom may number number see attached file hp...
4,0,hpl nom june number number see attached file h...
...,...,...
82481,1,info advantageapartmentscom infoadvantageapart...
82482,1,monkeyorg helpdeskmonkeyorg monkeyorg hi josep...
82483,1,help center infohelpcentercoza_infohelpcenterc...
82484,1,metamask infosofamekarcom verify metamask wall...


In [17]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test = train_test_split(df['clean_text'],df['label'],test_size=0.2,random_state=42)

In [19]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    stop_words='english'
)

In [22]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import StackingClassifier,RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline

In [25]:
base_para = [
    ('lr', LogisticRegression(C=2.0, max_iter=1000, random_state=42)),
    ('rf', RandomForestClassifier(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1)),
    ('svm',CalibratedClassifierCV(LinearSVC(C=1.0, random_state=42, max_iter=2000)))
]

stack = StackingClassifier(estimators=base_para,final_estimator=XGBClassifier(n_estimators=100,learning_rate=0.1,max_depth=4,random_state=42,eval_matrics='logloss'),cv=5,n_jobs=-1)

In [27]:
model_pipeline = Pipeline([
    ('tfidf', tfidf),
    ('stacking', stack)
])

In [28]:
model_pipeline.fit(x_train,y_train)

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [17:22:23] WARNING: /__w/xgboost/xgboost/src/learner.cc:793: 
Parameters: { "eval_matrics" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Pipeline(steps=[('tfidf',
                 TfidfVectorizer(max_features=10000, ngram_range=(1, 2),
                                 stop_words='english')),
                ('stacking',
                 StackingClassifier(cv=5,
                                    estimators=[('lr',
                                                 LogisticRegression(C=2.0,
                                                                    max_iter=1000,
                                                                    random_state=42)),
                                                ('rf',
                                                 RandomForestClassifier(max_depth=15,
                                                                        n_jobs=-1,
                                                                        random_state=42)),
                                                ('svm',
                                                 CalibratedClassifierCV(estimator=LinearSVC(max_iter=2000,
                                                                                            rand...
                                                                  feature_weights=None,
                                                                  gamma=None,
                                                                  grow_policy=None,
                                                                  importance_type=None,
                                                                  interaction_constraints=None,
                                                                  learning_rate=0.1,
                                                                  max_bin=None,
                                                                  max_cat_threshold=None,
                                                                  max_cat_to_onehot=None,
                                                                  max_delta_step=None,
                                                                  max_depth=4,
                                                                  max_leaves=None,
                                                                  min_child_weight=None,
                                                                  missing=nan,
                                                                  monotone_constraints=None,
                                                                  multi_strategy=None,
                                                                  n_estimators=100,
                                                                  n_jobs=None, ...),
                                    n_jobs=-1))])

In [29]:
y_pred = model_pipeline.predict(x_test)
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix
print(accuracy_score(y_test,y_pred))
print(confusion_matrix(y_test,y_pred))
print(classification_report(y_test,y_pred))

0.9864225966783853
[[7818  117]
 [ 107 8456]]
              precision    recall  f1-score   support

           0       0.99      0.99      0.99      7935
           1       0.99      0.99      0.99      8563

    accuracy                           0.99     16498
   macro avg       0.99      0.99      0.99     16498
weighted avg       0.99      0.99      0.99     16498



In [32]:
import pickle
pickle_filename = "phishingdetect.pkl"

with open(pickle_filename, "wb") as f:
    pickle.dump(model_pipeline, f)

check if it is working

In [33]:
with open("phishingdetect.pkl", "rb") as f:
    loaded_model = pickle.load(f)

sample_email = ["URGENT: Your bank account was accessed from an unrecognized device. Click here to verify your identity."]
clean_sample = [clean_email_text(sample_email[0])]

prediction = loaded_model.predict(clean_sample)[0]
confidence = loaded_model.predict_proba(clean_sample)[0][prediction]

label_text = "PHISHING" if prediction == 1 else "SAFE"
print(f"Prediction Result: {label_text} (Confidence: {confidence * 100:.2f}%)")

Prediction Result: PHISHING (Confidence: 99.99%)
